### Loading & Testing NLP Model

In [1]:
import requests
import spacy
from spacytextblob.spacytextblob import SpacyTextBlob
import json
import pandas as pd

In [ ]:
# Download the spacy language model if it's not already installed
try:
    nlp = spacy.load("en_core_web_md")
except OSError:
    print("Downloading spaCy model 'en_core_web_md'...")
    nlp = spacy.load("en_core_web_md")

In [3]:
nlp.add_pipe("spacytextblob")

doc = nlp("Argentina faces severe economic crisis")
print(doc._.blob.polarity)  # returns -1 to +1

0.2


### Testing Natural Language Processing (NLP)

In [10]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyser = SentimentIntensityAnalyzer()

def sentiment_score(text):

    scores = analyser.polarity_scores(text)
    compound = scores["compound"]

    if compound <= -0.6:
        label = "Very negative"
        score = 1
    elif compound <= -0.2:
        label = "Negative"
        score = 2
    elif compound < 0.2:
        label = "Neutral"
        score = 3
    elif compound < 0.6:
        label = "Positive"
        score = 4
    else:
        label = "Very positive"
        score = 5

    return {
        "label": label,
        "score": score,
        "compound": round(compound, 3),
        "details": scores
    }

example_headlines = ["UK borrowing costs hit highest since 2008 financial crisis. The interest rate on government debt is climbing over fears about inflation, interest rates",
                     "Stock markets surge and oil tumbles as Trump postpones power plant strikes after 'very good and productive' talks with Iran"]

for headline in example_headlines:
  print(headline)
  print(sentiment_score(headline))
  print("\n")

UK borrowing costs hit highest since 2008 financial crisis. The interest rate on government debt is climbing over fears about inflation, interest rates
{'label': 'Negative', 'score': 2, 'compound': -0.527, 'details': {'neg': 0.281, 'neu': 0.539, 'pos': 0.18, 'compound': -0.5267}}


Stock markets surge and oil tumbles as Trump postpones power plant strikes after 'very good and productive' talks with Iran
{'label': 'Neutral', 'score': 3, 'compound': -0.104, 'details': {'neg': 0.186, 'neu': 0.686, 'pos': 0.129, 'compound': -0.1045}}




### Creating & Populating the Database

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv
import os
import sqlite3
from pathlib import Path

load_dotenv()

GUARDIAN_API_KEY = os.getenv("GUARDIAN_API_KEY")

countries = ['Turkey', 'Nigeria', 'Argentina', 'India', 'Vietnam']

start_date = datetime(2026, 1, 1)
end_date = datetime(2026, 1, 30)

# Database location
DB_PATH = Path("../data/articles.db")
DB_PATH.parent.mkdir(parents=True, exist_ok=True)


def create_database(db_path):
    """
    Create the SQLite database, articles table, and indexes.
    """
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS articles (
        article_id TEXT PRIMARY KEY,
        country TEXT,
        date TEXT,
        source_name TEXT,
        title TEXT,
        description TEXT,
        url TEXT,
        published_at TEXT
    )
    """)

    cursor.execute("CREATE INDEX IF NOT EXISTS idx_country ON articles(country)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_date ON articles(date)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_published_at ON articles(published_at)")

    conn.commit()
    return conn, cursor


# Create database connection
conn, cursor = create_database(DB_PATH)

all_articles_data = []
current_date = start_date

while current_date <= end_date:
    date_str = current_date.strftime('%Y-%m-%d')

    print(f"Fetching articles for date: {date_str}")

    for country in countries:
        page = 1

        while True:
            url = "https://content.guardianapis.com/search"

            params = {
                "q": country,
                "section": "business",
                "from-date": date_str,
                "to-date": date_str,
                "api-key": GUARDIAN_API_KEY,
                "page-size": 50,
                "page": page,
                "show-fields": "headline,trailText"
            }

            response = requests.get(url, params=params)

            if response.status_code != 200:
                print(
                    f"API request failed for {country} on {date_str}, page {page}: "
                    f"{response.status_code} - {response.text}"
                )
                print("Waiting for 60 seconds before retrying...")
                time.sleep(60)
                continue

            data = response.json()

            if "response" not in data or "results" not in data["response"]:
                print(f"Unexpected API response for {country} on {date_str}, page {page}: {data}")
                break

            results = data["response"]["results"]

            if not results:
                break

            for article in results:
                article_id = article.get("id")

                if not article_id:
                    continue

                # Check whether article already exists in database
                cursor.execute(
                    "SELECT 1 FROM articles WHERE article_id = ?",
                    (article_id,)
                )

                already_exists = cursor.fetchone()

                if already_exists:
                    print(f"Already cached in database: {article_id}")
                    continue

                fields = article.get("fields", {})

                article_data = {
                    "article_id": article_id,
                    "country": country,
                    "date": date_str,
                    "source_name": "The Guardian",
                    "title": fields.get("headline"),
                    "description": fields.get("trailText"),
                    "url": article.get("webUrl"),
                    "published_at": article.get("webPublicationDate")
                }

                cursor.execute(
                    """
                    INSERT OR IGNORE INTO articles (
                        article_id,
                        country,
                        date,
                        source_name,
                        title,
                        description,
                        url,
                        published_at
                    )
                    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                    """,
                    (
                        article_data["article_id"],
                        article_data["country"],
                        article_data["date"],
                        article_data["source_name"],
                        article_data["title"],
                        article_data["description"],
                        article_data["url"],
                        article_data["published_at"]
                    )
                )

                conn.commit()

                print(f"Inserted into database: {article_id}")

                all_articles_data.append(article_data)

            if page >= data["response"]["pages"]:
                break

            page += 1

    current_date += timedelta(days=1)

Fetching articles for date: 2026-01-01
Fetching articles for date: 2026-01-02
Already cached in database: business/live/2026/jan/02/uk-house-prices-drop-december-lidl-christmas-sales-ftse-business-live-news
Fetching articles for date: 2026-01-03
Fetching articles for date: 2026-01-04
Fetching articles for date: 2026-01-05
Fetching articles for date: 2026-01-06
Already cached in database: business/2026/jan/06/jaguar-land-rover-sales-us-tariffs-cyber-attack-shares-tata-motors
Already cached in database: business/live/2026/jan/06/jlr-sales-hit-by-cyber-attack-next-beats-christmas-sales-expectations-uk-services-sector-stock-market-business-live
Already cached in database: business/2026/jan/06/bank-of-england-venezuelan-gold-nicolas-maduro-us-uk
Fetching articles for date: 2026-01-07
Fetching articles for date: 2026-01-08
Already cached in database: business/2026/jan/08/shadow-fleet-ships-sanctioned-oil-reflagged-russia-lloyds-list-analysis
Already cached in database: business/live/2026/jan

### Querying the SQLite Database

In [59]:
conn = sqlite3.connect("../data/articles.db")

query = "SELECT * FROM articles"
df_headlines = pd.read_sql_query(query, conn)
print(df_headlines)

conn.close()

                                           article_id    country        date  \
0   business/live/2026/jan/02/uk-house-prices-drop...      India  2026-01-02   
1   business/2026/jan/06/jaguar-land-rover-sales-u...      India  2026-01-06   
2   business/live/2026/jan/06/jlr-sales-hit-by-cyb...      India  2026-01-06   
3   business/2026/jan/06/bank-of-england-venezuela...    Vietnam  2026-01-06   
4   business/2026/jan/08/shadow-fleet-ships-sancti...     Turkey  2026-01-08   
5   business/live/2026/jan/08/marks-spencer-tesco-...     Turkey  2026-01-08   
6   business/live/2026/jan/09/rio-tinto-glencore-m...  Argentina  2026-01-09   
7   business/2026/jan/09/high-costs-falling-return...      India  2026-01-09   
8   business/2026/jan/10/russia-economy-collapse-o...     Turkey  2026-01-10   
9   business/2026/jan/11/trump-move-for-venezuelas...     Turkey  2026-01-11   
10  business/2026/jan/11/game-on-the-swiss-sports-...    Vietnam  2026-01-11   
11  business/2026/jan/12/trump-fed-us-ce